In [15]:
!pip install selenium

In [16]:
import csv
import json
import logging
import random
import re
import time
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

BASE_URL = "https://store.steampowered.com/search?term="
HEADERS = {"User-Agent": "Mozilla/5.0"}

session = requests.Session()
session.headers.update(HEADERS)

def load_config(filename="config.json"):
    with open(filename, "r", encoding="utf-8") as file:
        config = json.load(file)
    return config

# источники для логгирования: 
# https://habr.com/ru/companies/wunderfund/articles/683880/
# https://stackoverflow.com/questions/69291738/how-to-encode-all-logged-messages-as-utf-8-in-python

def setup_logging(config): 
    logger = logging.getLogger("steam_logger")
    logger.handlers.clear()

    if not config["logging_enabled"]:
        logger.disabled = True
        return logger

    logger.disabled = False

    level_name = config["logging_level"].upper()
    level = getattr(logging, level_name, logging.INFO)

    logger.setLevel(level)

    file_handler = logging.FileHandler(config["log_file"], mode="w", encoding="utf-8")
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    file_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    return logger

config = load_config("config.json")
logger = setup_logging(config)

# Источники:
# https://timeweb.cloud/tutorials/python/selenium-parsing-dinamicheskih-sajtov
# https://happypython.ru/2022/11/27/selenium-chromedriver-find-elements/

def get_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(options=options)
    return driver

driver = get_driver()

def get_soup_with_requests(url):
    response = session.get(url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    return soup

def get_soup_with_selenium(url):
    driver.get(url)
    time.sleep(2) # даем странице прогрузиться
    soup = BeautifulSoup(driver.page_source, "lxml")
    return soup

def get_game_links(page):
    url = BASE_URL.format(page)
    logger.info(f"Открываем страницу поиска: {url}")

    try:
        soup = get_soup_with_requests(url)
    except requests.RequestException as error:
        logger.warning(f"Requests не сработал, пробуем selenium: {url} | {error}")
        soup = get_soup_with_selenium(url)

    links = []

    for tag in soup.select("a.search_result_row"):
        href = tag.get("href")
        if href:
            clean_link = href.split("?")[0]
            links.append(clean_link)

    logger.info(f"На странице {page} найдено {len(links)} ссылок")
    return links

def to_int(text):
    if not text:
        return None

    text = re.sub(r"[^\d]", "", text) # оставляем только цифры

    if text == "":
        return None

    return int(text)

def parse_reviews(soup):
    text = soup.get_text("\n", strip=True)

    reviews = None
    total_reviews = None
    total_positive = None
    total_negative = None

    review_block = soup.select_one("div.user_reviews_summary_row span.game_review_summary")
    if review_block:
        reviews = review_block.get_text(strip=True)

    if not reviews:
        match = re.search(
            r"Total reviews in all languages:\s*[\d,.\s]+\s*([A-Za-z ]+)",
            text
        )
        if match:
            reviews = match.group(1).strip()

    match = re.search(r"All\s*\(([\d,.\s]+)\)", text)
    if match:
        total_reviews = to_int(match.group(1))
    else:
        match = re.search(r"Total reviews in all languages:\s*([\d,.\s]+)", text)
        if match:
            total_reviews = to_int(match.group(1))

    positive_list = re.findall(r"Positive\s*\(([\d,.\s]+)\)", text)
    negative_list = re.findall(r"Negative\s*\(([\d,.\s]+)\)", text)

    positive_list = [to_int(x) for x in positive_list if to_int(x) is not None]
    negative_list = [to_int(x) for x in negative_list if to_int(x) is not None]

    if total_reviews is not None:
        for pos in positive_list:
            for neg in negative_list:
                if pos + neg == total_reviews: # ищем нужную пару
                    total_positive = pos
                    total_negative = neg
                    break
            if total_positive is not None:
                break

    return {
        "reviews": reviews,
        "total_reviews": total_reviews,
        "total_positive": total_positive,
        "total_negative": total_negative,
    }

def parse_system_requirements(soup):
    text = soup.get_text("\n", strip=True)

    match = re.search(
        r"System Requirements\s+(.*?)\s+(?:What Curators Say|Customer reviews for|Mature Content Description|$)",
        text,
        re.S
    )

    if not match:
        return None

    requirements = re.sub(r"\s+", " ", match.group(1)).strip()
    return requirements

def parse_game(url):
    try:
        logger.info(f"Парсим игру: {url}")

        try:
            soup = get_soup_with_requests(url)
        except requests.RequestException as error:
            logger.warning(f"Requests не сработал для игры, пробуем selenium: {url} | {error}")
            soup = get_soup_with_selenium(url)

        name_tag = soup.select_one("div.apphub_AppName")
        if name_tag:
            name = name_tag.get_text(strip=True)
        else:
            name = None

        description_tag = soup.select_one("div.game_description_snippet")
        if description_tag:
            description = description_tag.get_text(strip=True)
        else:
            description = None

        price_tag = soup.select_one("div.game_purchase_price")
        if price_tag is None:
            price_tag = soup.select_one("div.discount_final_price")

        if price_tag:
            price = price_tag.get_text(strip=True)
        else:
            price = "Free/Unknown"

        tags = []
        for tag in soup.select("a.app_tag"):
            text = tag.get_text(strip=True)
            if text:
                tags.append(text)

        review_data = parse_reviews(soup)
        system_requirements = parse_system_requirements(soup)

        game_info = {
            "name": name,
            "description": description,
            "price": price,
            "reviews": review_data["reviews"],
            "total_reviews": review_data["total_reviews"],
            "total_positive": review_data["total_positive"],
            "total_negative": review_data["total_negative"],
            "system_requirements": system_requirements,
            "tags": ",".join(tags),
            "url": url,
        }

        logger.info(f"Игра сохранена: {name}")
        return game_info

    except requests.Timeout:
        logger.warning(f"Таймаут: {url}")
        return None
    except requests.HTTPError as error:
        logger.error(f"HTTP ошибка: {url} | {error}")
        return None
    except requests.RequestException as error:
        logger.error(f"Ошибка запроса: {url} | {error}")
        return None
    except Exception as error:
        logger.exception(f"Другая ошибка: {url} | {error}")
        return None

def save_csv_part(data, filename="steam_games.csv"):
    if not data:
        return

    try:
        file_exists = False

        try:
            with open(filename, "r", encoding="utf-8"):
                file_exists = True
        except FileNotFoundError:
            file_exists = False

        with open(filename, "a", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=data[0].keys())

            if not file_exists:
                writer.writeheader() # заголовок пишем только один раз

            writer.writerows(data)

        logger.info(f"Сохранено {len(data)} строк в {filename}")
    except Exception as error:
        logger.exception(f"Ошибка при сохранении CSV: {error}")

def scrape(limit=100, save_every=1000, filename="steam_games.csv"):
    all_games = []
    buffer_data = []
    seen_links = set()
    page = 1

    logger.info(f"Старт. Нужно собрать {limit} игр")

    while len(all_games) < limit:
        print(f"Page {page}")
        logger.info(f"Идет страница {page}")

        try:
            links = get_game_links(page)
        except Exception as error:
            logger.error(f"Не удалось открыть страницу {page}: {error}")
            break

        if not links:
            logger.warning(f"На странице {page} нет ссылок")
            break

        for link in links:
            if len(all_games) >= limit:
                break

            if link in seen_links:
                continue

            seen_links.add(link)
            print(link)

            game = parse_game(link)
            if game is not None:
                all_games.append(game)
                buffer_data.append(game)

            if len(buffer_data) >= save_every:
                save_csv_part(buffer_data, filename)
                buffer_data = []

            time.sleep(random.uniform(0.5, 1.2)) # пауза между запросами

        page += 1

    if buffer_data:
        save_csv_part(buffer_data, filename) # сохраняем остаток

    logger.info(f"Конец. Собрано {len(all_games)} игр")
    return all_games

data = scrape(limit=20, save_every=10, filename="steam_games.csv")

driver.quit()

print("Done:", len(data))

Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


Page 1
https://store.steampowered.com/app/730/CounterStrike_2/
https://store.steampowered.com/app/3321460/Crimson_Desert/
https://store.steampowered.com/app/230410/Warframe/
https://store.steampowered.com/app/2868840/Slay_the_Spire_2/
https://store.steampowered.com/app/578080/PUBG_BATTLEGROUNDS/
https://store.steampowered.com/app/1172470/Apex_Legends/
https://store.steampowered.com/app/3564740/Where_Winds_Meet/
https://store.steampowered.com/app/1808500/ARC_Raiders/
https://store.steampowered.com/app/236390/War_Thunder/
https://store.steampowered.com/app/570/Dota_2/
https://store.steampowered.com/app/252490/Rust/
https://store.steampowered.com/app/3405690/EA_SPORTS_FC_26/
https://store.steampowered.com/app/227300/Euro_Truck_Simulator_2/
https://store.steampowered.com/app/2183900/Warhammer_40000_Space_Marine_2/
https://store.steampowered.com/app/2139460/Once_Human/
https://store.steampowered.com/app/3679080/The_Seven_Deadly_Sins_Origin/
https://store.steampowered.com/app/1449850/YuGiOh_